In [ ]:
import json

from pygamma_agreement import Continuum
from pygamma_agreement import CombinedCategoricalDissimilarity
from pyannote.core import Segment
from itertools import combinations

### Préparation des données

In [ ]:
files = {
    "annotation1": "annotation1.json",
    "annotation2":"annotation2.json",
}

continuum = Continuum()

OFFSET_SHIFT = 1_000_000
for annotator_name, file_path in files.items():
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
        # On trie par ID de tâche pour être sûr de l'ordre
        for idx, task in enumerate(data):
            task_offset = idx * OFFSET_SHIFT
            
            # Label Studio stocke les annotations dans 'annotations'
            # On prend la première (puisque chaque fichier est propre à UN annotateur)
            if not task.get('annotations'):
                continue
                
            results = task['annotations'][0].get('result', [])
            
            for res in results:
                if res['type'] == 'labels':
                    start = float(res['value']['start']) + task_offset
                    end = float(res['value']['end']) + task_offset
                    
                    # Sécurité : pygamma ne supporte pas les segments de durée nulle
                    if end > start:
                        label = res['value']['labels'][0]
                        # On passe l'objet Segment de pyannote.core
                        continuum.add(annotator_name, Segment(start, end), label)

# Calcul du Gamma
dissimilarity = CombinedCategoricalDissimilarity()
gamma_results = continuum.compute_gamma(dissimilarity)

print(f"Accord inter-annotateur (Gamma) : {gamma_results.gamma:.3f}")


In [ ]:
annotateurs = list(continuum.annotators)
paires = list(combinations(annotateurs, 2))

print("--- Accords Pair-à-Pair ---")
for a1, a2 in paires:
    # 1. Créer un nouveau continuum vide pour le duo
    sub_continuum = Continuum()
    
    # 2. Copier les annotations des deux annotateurs cibles
    # On accède directement au dictionnaire interne des annotations
    for annotateur in [a1, a2]:
        for unit in continuum.iter_annotator(annotateur):
            # unit contient généralement (segment, annotation)
            sub_continuum.add(annotateur, unit.segment, unit.annotation)
    
    # 3. Calcul du Gamma pour ce duo
    try:
        res = sub_continuum.compute_gamma(dissimilarity)
        print(f"Accord entre {a1} et {a2} : {res.gamma:.3f}")
    except Exception as e:
        print(f"Erreur pour la paire {a1}-{a2} : {e}")